# Phase 3: Reinforcement Learning — Tabular Q-Learning & DQN

This notebook trains and evaluates two reinforcement learning policies on the **SmartRetail Customer Recommendation Environment**:

| Policy | Description |
|---|---|
| **Tabular Q-Learning** | 8-cluster discretized state space, ε-greedy exploration, 500 episodes |
| **Deep Q-Network (DQN)** | PyTorch QNetwork with [32,16] hidden layers, experience replay, 300 episodes |

Baselines compared at evaluation time:
- **Always No-Action** — never sends an offer
- **Random Action** — uniformly random policy

### Action Space
| Action | Description | Cost |
|---|---|---|
| 0 | No Action | \$0.00 |
| 1 | 10% Discount Coupon | \$1.00 |
| 2 | Free Premium Trial | \$5.00 |

### State Space
Continuous 5-D customer state vector: `[Recency, Frequency, Monetary, PC1, PC2]`  
Discretized into **8 clusters** via KMeans for the tabular agent.

In [ ]:
# Ensure project root is on the path when running from notebooks/
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.rcParams['figure.dpi'] = 120
print('All imports successful.')

---
## 1. Load Modules and Configuration

In [ ]:
from src.config import get_absolute_path, config_data
from src.environment import RetailCustomerEnv
from src.rl_agents import StateDiscretizer, QLearningAgent, DQNAgent

rl_cfg = config_data['reinforcement_learning']
q_cfg  = rl_cfg['tabular_q']
dqn_cfg = rl_cfg['dqn']

actions_desc = rl_cfg['actions']
costs        = {int(k): v for k, v in rl_cfg['costs'].items()}
adjustments  = {int(k): v for k, v in rl_cfg['adjustments'].items()}

print('Config loaded successfully.')
print(f"  Tabular Q: {q_cfg['episodes']} episodes | alpha={q_cfg['alpha']} | gamma={q_cfg['gamma']}")
print(f"  DQN:       {dqn_cfg['episodes']} episodes | lr={dqn_cfg['learning_rate']} | gamma={dqn_cfg['gamma']}")

---
## 2. Initialise the Customer Environment

In [ ]:
env_train = RetailCustomerEnv(split='train')

print(f'Training customers : {env_train.num_customers:,}')
print(f'State dimensions   : {env_train.states.shape[1]}')
print(f'State sample (first customer):')
print(f'  [Recency, Frequency, Monetary, PC1, PC2] = {env_train.states[0].round(3)}')

---
## 3. Fit K-Means State Discretizer

In [ ]:
N_CLUSTERS = int(rl_cfg['state_clusters'])

discretizer = StateDiscretizer(n_clusters=N_CLUSTERS, random_state=42)
discretizer.fit(env_train.states)

# Pre-discretize all training states for speed
train_state_indices = discretizer.kmeans.predict(env_train.states)

unique, counts = np.unique(train_state_indices, return_counts=True)
print(f'KMeans fitted with {N_CLUSTERS} clusters.')
print('Cluster distribution:')
for c, n in zip(unique, counts):
    print(f'  Cluster {c}: {n:,} customers ({100*n/len(train_state_indices):.1f}%)')

---
## 4. Train Tabular Q-Learning Agent (500 Episodes)

In [ ]:
q_agent = QLearningAgent(
    state_size=N_CLUSTERS,
    action_size=3,
    alpha=float(q_cfg['alpha']),
    gamma=float(q_cfg['gamma']),
    epsilon=float(q_cfg['epsilon_start']),
    epsilon_decay=float(q_cfg['epsilon_decay']),
    epsilon_min=float(q_cfg['epsilon_min'])
)

Q_EPISODES = int(q_cfg['episodes'])
q_rewards  = []

for ep in range(1, Q_EPISODES + 1):
    env_train.reset()
    total_reward = 0.0
    for idx in range(env_train.num_customers):
        s_idx    = train_state_indices[idx]
        action   = q_agent.get_action(s_idx, train=True)
        _, r, done, _ = env_train.step(action)
        ns_idx   = 0 if done else train_state_indices[idx + 1]
        q_agent.update(s_idx, action, r, ns_idx)
        total_reward += r
    q_agent.decay_epsilon()
    q_rewards.append(total_reward)
    if ep % 100 == 0 or ep == 1:
        print(f'  Episode {ep:4d}/{Q_EPISODES} | Reward: {total_reward:10,.2f} | ε={q_agent.epsilon:.3f}')

q_agent.save()
print('\nTabular Q-Learning training complete. Model saved.')

---
## 5. Train Deep Q-Network Agent (300 Episodes)

In [ ]:
dqn_agent = DQNAgent(
    state_size=5,
    action_size=3,
    lr=float(dqn_cfg['learning_rate']),
    gamma=float(dqn_cfg['gamma']),
    epsilon=float(dqn_cfg['epsilon_start']),
    epsilon_decay=float(dqn_cfg['epsilon_decay']),
    epsilon_min=float(dqn_cfg['epsilon_min']),
    buffer_size=int(dqn_cfg['buffer_size']),
    batch_size=int(dqn_cfg['batch_size'])
)

DQN_EPISODES   = int(dqn_cfg['episodes'])
TARGET_UPDATE  = int(dqn_cfg['target_update_frequency'])
dqn_rewards    = []

for ep in range(1, DQN_EPISODES + 1):
    state = env_train.reset()
    total_reward = 0.0
    done = False
    step_count = 0
    while not done:
        action = dqn_agent.get_action(state, train=True)
        next_state, reward, done, _ = env_train.step(action)
        dqn_agent.memory.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        step_count += 1
        if step_count % 16 == 0:
            dqn_agent.update()
    dqn_agent.decay_epsilon()
    dqn_rewards.append(total_reward)
    if ep % TARGET_UPDATE == 0:
        dqn_agent.update_target_network()
    if ep % 60 == 0 or ep == 1:
        print(f'  Episode {ep:4d}/{DQN_EPISODES} | Reward: {total_reward:10,.2f} | ε={dqn_agent.epsilon:.3f}')

dqn_agent.save()
print('\nDQN training complete. Model saved.')

---
## 6. Plot Training Reward Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Reinforcement Learning — Training Reward Curves', fontsize=14, fontweight='bold')

# Smooth helper
def smooth(arr, window=20):
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='valid')

# Tabular Q
ax = axes[0]
ax.plot(q_rewards, alpha=0.3, color='#10b981', linewidth=0.8, label='Raw')
ax.plot(range(19, len(q_rewards)), smooth(q_rewards), color='#10b981', linewidth=2, label='Smoothed (w=20)')
ax.set_title('Tabular Q-Learning (500 Episodes)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Episode Reward ($)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

# DQN
ax = axes[1]
ax.plot(dqn_rewards, alpha=0.3, color='#2563eb', linewidth=0.8, label='Raw')
if len(dqn_rewards) >= 20:
    ax.plot(range(19, len(dqn_rewards)), smooth(dqn_rewards), color='#2563eb', linewidth=2, label='Smoothed (w=20)')
ax.set_title('Deep Q-Network (300 Episodes)')
ax.set_xlabel('Episode')
ax.set_ylabel('Total Episode Reward ($)')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.5)

figures_dir = get_absolute_path('figures_dir')
os.makedirs(figures_dir, exist_ok=True)
curve_path = os.path.join(figures_dir, 'rl_reward_learning_curve.png')
plt.tight_layout()
plt.savefig(curve_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Learning curves saved to: {curve_path}')

---
## 7. Evaluate All Policies on Held-Out Test Split

In [ ]:
env_test = RetailCustomerEnv(split='test')
print(f'Test customers: {env_test.num_customers:,}')

def run_policy(env, policy_fn):
    """Run a callable policy_fn(state) -> action over all test customers."""
    env.reset()
    total = 0.0
    for i in range(env.num_customers):
        state  = env.states[i]
        action = policy_fn(state)
        _, r, _, _ = env.step(action)
        total += r
    return total

# Load saved agents to ensure we evaluate the serialised versions
q_eval  = QLearningAgent(state_size=N_CLUSTERS, action_size=3); q_eval.load()
dqn_eval = DQNAgent(state_size=5, action_size=3);                 dqn_eval.load()
disc_eval = StateDiscretizer(n_clusters=N_CLUSTERS);              disc_eval.load()

np.random.seed(42)

profits = {
    'Always No-Action' : run_policy(env_test, lambda s: 0),
    'Random Action'    : run_policy(env_test, lambda s: np.random.choice([0, 1, 2])),
    'Tabular Q-Learning': run_policy(env_test, lambda s: q_eval.get_action(disc_eval.discretize(s), train=False)),
    'DQN Recommendation': run_policy(env_test, lambda s: dqn_eval.get_action(s, train=False)),
}

print('\n=== Test Set Policy Evaluation Results ===')
for name, profit in profits.items():
    print(f'  {name:<22}: ${profit:>12,.2f}')

---
## 8. Comparative Profit Bar Chart

In [ ]:
import json

labels  = list(profits.keys())
values  = list(profits.values())
colors  = ['#ef4444', '#f59e0b', '#10b981', '#2563eb']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(labels, values, color=colors, edgecolor='black', alpha=0.88, width=0.55)

# Annotate bars
max_v = max(values)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            v + max_v * 0.012,
            f'${v:,.0f}',
            ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.set_ylabel('Total Cumulative Profit ($)', fontsize=12)
ax.set_title('RL Policy Profit Comparison on Held-Out Test Customers', fontsize=13, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.set_ylim(0, max_v * 1.15)

# Colour legend
legend_patches = [mpatches.Patch(facecolor=c, edgecolor='black', label=l)
                  for c, l in zip(colors, labels)]
ax.legend(handles=legend_patches, loc='upper left', fontsize=9)

plt.tight_layout()
bar_path = os.path.join(figures_dir, 'rl_profit_comparison.png')
plt.savefig(bar_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Profit comparison chart saved to: {bar_path}')

# Persist as JSON for Streamlit dashboard
processed_dir = get_absolute_path('processed_data_dir')
json_key_map = {'Always No-Action': 'Always_No_Action',
                'Random Action': 'Random_Action',
                'Tabular Q-Learning': 'Tabular_Q_Policy',
                'DQN Recommendation': 'DQN_Policy'}
serialisable = {json_key_map[k]: v for k, v in profits.items()}
with open(os.path.join(processed_dir, 'rl_evaluation_results.json'), 'w') as f:
    json.dump(serialisable, f, indent=4)
print('Evaluation JSON saved.')

---
## 9. Inspect Learned Q-Table (Tabular Agent)

Each row is a KMeans cluster (state). Each column is an action:  
`0 = No Action | 1 = 10% Discount | 2 = Free Premium Trial`

In [ ]:
import pandas as pd

q_table = q_eval.q_table
action_cols = [f'Q(s,{a}) — {actions_desc[str(a)]}' for a in range(3)]
q_df = pd.DataFrame(q_table, columns=action_cols)
q_df.index.name = 'Cluster (State)'
q_df['Best Action'] = q_df.idxmax(axis=1).str.extract(r'Q\(s,(\d)\)').astype(int)
q_df['Best Action Label'] = q_df['Best Action'].map(lambda a: actions_desc[str(a)])

print('Learned Q-Table:')
display(q_df.round(4))

---
## Summary

| Agent | Architecture | Episodes | Serialised Model |
|---|---|---|---|
| Tabular Q-Learning | 8×3 Q-table | 500 | `models/q_table.npy` |
| Deep Q-Network | FC [5→32→16→3] + Target Net | 300 | `models/dqn_model.pt` |

Both agents and the KMeans discretizer are serialised to `models/` and are loaded by the Streamlit dashboard (`app.py`) for real-time customer recommendation inference.